# Predictive Optimization of Treasury & Liquidity Flows — STB Bank

Internship project (Data/BI Analyst intern, STB Bank, Summer 2025).
Builds predictive models (XGBoost, Random Forest) on a dimensional
(star-schema) dataset of daily cash-register balances to forecast
branch-level liquidity, feeding a Power BI dashboard for treasury
monitoring.

**Note:** the real STB Bank data this notebook was developed and run
against (daily cash balances, branch/register dimension tables, the
Power BI file) is internal banking data covered by confidentiality and
is intentionally **not included** in this repository. See the README
for the full write-up, methodology, and expected input schema so the
pipeline can be reproduced against equivalent data.


**Data Loading and Cleaning**

Imports pandas and loads data from four CSV files into DataFrames: `fact`, `dim_caisse`, `dim_agence`, and `dim_date`. Removes quotes from column names in `dim_date`.


In [ ]:
import pandas as pd
fact = pd.read_csv("FactSoldeCaisse_final_cleaned.csv")
dim_caisse = pd.read_csv("DimCaisse_cleaned.csv")
dim_agence = pd.read_csv("DimAgences_cleaned_presentable.csv")
dim_date = pd.read_csv("dimdate.csv")

# Remove quotes from column names
dim_date.columns = dim_date.columns.str.replace('"', '')

In [ ]:
fact

In [ ]:
dim_agence

In [ ]:
dim_caisse

In [ ]:
dim_date

In [ ]:
if len(dim_date.columns) == 1:
    dim_date = dim_date[dim_date.columns[0]].str.split(',', expand=True)
    dim_date.columns = ['date', 'annee', 'mois', 'jour', 'jour_semaine', 'est_weekend', 'est_jour_ferie', 'nom_jour_ferie']

In [ ]:
import pandas as pd

dim_date = pd.read_csv('dimdate.csv', sep=',')

# If only one column, split it manually
if len(dim_date.columns) == 1:
    dim_date = dim_date[dim_date.columns[0]].str.split(',', expand=True)
    dim_date.columns = ['date', 'annee', 'mois', 'jour', 'jour_semaine', 'est_weekend', 'est_jour_ferie', 'nom_jour_ferie']

print(repr(dim_date.columns.tolist()))  # Should now show separate columns

# Now you can use the 'date' column
dim_date['date'] = pd.to_datetime(dim_date['date'])
print(dim_date.head())

In [ ]:
dim_date['date'] = pd.to_datetime(dim_date['date'])

In [ ]:
print(repr(dim_date.columns.tolist()))

In [ ]:
import pandas as pd

# Ensure date columns are datetime
fact['Date_Position'] = pd.to_datetime(fact['Date_Position'])
dim_date['date'] = pd.to_datetime(dim_date['date'])

# 1. Merge fact with dim_caisse on 'Caisse_Key'
merged_caisse_fact = fact.merge(dim_caisse, on='Caisse_Key', how='inner')

# 2. Merge with dim_agence on 'Code_Agence' (ensure correct dtype)
merged_caisse_fact['Code_Agence'] = merged_caisse_fact['Code_Agence'].astype(int)
merged_df = merged_caisse_fact.merge(dim_agence, on='Code_Agence', how='inner', suffixes=('', '_agence'))

# 3. Merge with dim_date on Date_Position == date
# Rename before merge to avoid name conflict
merged_df.rename(columns={'Date_Position': 'fact_Date_Position'}, inplace=True)
merged_df = merged_df.merge(
    dim_date,
    left_on='fact_Date_Position',
    right_on='date',
    how='left'
)

# 4. (Optional) Drop duplicate 'date' column if needed
# merged_df.drop('date', axis=1, inplace=True)

# 5. Output preview
print(merged_df.head())
print(merged_df.columns)

In [ ]:
columns_to_drop = [
    'Code_Caisse_x',
    'Code_Devise',
    'Date_Fin_Position',
    'Solde',
    'Position_Duration',
    'Code_Caisse_y',
    'Code_Caisse_Chef_Lieu',
    'Type_Caisse',
    'Box_de_Change',
    'Code_DR',
    'DR'
]

merged_df_cleaned = merged_df.drop(columns=columns_to_drop, errors='ignore')

print(merged_df_cleaned.head())

In [ ]:
columns_to_drop_further = [
    'Agence',
    'Adresse_Key',
    'Latitude',
    'Longitude',
    'Code_Gov',
    'Code_Postal',
    'Matricule_ChefAgence',
    'Chef_Agence',
    'est_weekend', # Added this column to the list
    'est_jour_ferie', # Added this column to the list
    'nom_jour_ferie' # Added this column to the list
]

merged_df_cleaned = merged_df_cleaned.drop(columns=columns_to_drop_further, errors='ignore')

merged_df_cleaned

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select a random Caisse_Key
random_caisse_key = np.random.choice(merged_df_cleaned['Caisse_Key'].unique())
# Filter the dataframe for the selected Caisse_Key
caisse_df = merged_df_cleaned[merged_df_cleaned['Caisse_Key'] == random_caisse_key].copy()

# Sort by date to ensure correct plotting
caisse_df = caisse_df.sort_values(by='date')

# Plot Solde_TND over time
plt.figure(figsize=(12, 6))
sns.lineplot(data=caisse_df, x='date', y='Solde_TND')
plt.title(f'Solde_TND over Time for Caisse Key: {random_caisse_key}')
plt.xlabel('Date')
plt.ylabel('Solde_TND')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Ensure date column is datetime (already done in previous steps, but good to be safe)
merged_df_cleaned['date'] = pd.to_datetime(merged_df_cleaned['date'])

# Sort by Caisse_Key and date for correct feature calculation
merged_df_cleaned = merged_df_cleaned.sort_values(by=['Caisse_Key', 'date']).reset_index(drop=True)

# Calculate lag features based on Caisse_Key
merged_df_cleaned['lag_1'] = merged_df_cleaned.groupby('Caisse_Key')['Solde_TND'].shift(1)
merged_df_cleaned['lag_7'] = merged_df_cleaned.groupby('Caisse_Key')['Solde_TND'].shift(7)

# Calculate rolling features based on Caisse_Key
merged_df_cleaned['rolling_mean_3'] = merged_df_cleaned.groupby('Caisse_Key')['Solde_TND'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
merged_df_cleaned['rolling_std_7'] = merged_df_cleaned.groupby('Caisse_Key')['Solde_TND'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std()
)

# Display the head of the dataframe with the new features
print(merged_df_cleaned.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use your actual dataframe name here
df = merged_df_cleaned

# 1. Basic info and first rows
print("==== DataFrame Info ====")
print(df.info())
print("\n==== First 5 Rows ====")
print(df.head())

# 2. Unique values per column
print("\n==== Unique Values Per Column ====")
print(df.nunique())

# 3. Missing values per column
print("\n==== Missing Values Per Column ====")
print(df.isnull().sum())

# 4. Value counts for key categorical columns
categorical_cols = ['jour_semaine', 'est_weekend', 'est_jour_ferie', 'Code_Caisse', 'Code_Agence']
for col in categorical_cols:
    if col in df.columns:
        print(f"\n==== Value Counts for {col} ====")
        print(df[col].value_counts(dropna=False))

# 5. Distribution plots for key numerical columns
numerical_cols = ['Solde_TND', 'lag_1', 'lag_7', 'rolling_mean_3', 'rolling_std_7']
for col in numerical_cols:
    if col in df.columns:
        plt.figure(figsize=(6, 3))
        sns.histplot(df[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}')
        plt.show()

# 6. Correlation heatmap for numeric columns
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# 7. Scatter plot for spatial features (if present)
if 'Latitude' in df.columns and 'Longitude' in df.columns:
    plt.figure(figsize=(6, 6))
    plt.scatter(df['Longitude'], df['Latitude'], alpha=0.3)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Spatial Distribution of Agencies')
    plt.show()

In [ ]:
df_model=merged_df_cleaned.copy()
df_model

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Ensure date columns are datetime (already done, but for safety)
merged_df_cleaned['date'] = pd.to_datetime(merged_df_cleaned['date'])

# Sort by Caisse_Key and date for correct feature calculation (already done, but for safety)
merged_df_cleaned = merged_df_cleaned.sort_values(by=['Caisse_Key', 'date']).reset_index(drop=True)

# 1. Convert est_weekend and est_jour_ferie to integer (these columns were removed, skipping this step)
# merged_df_cleaned['est_weekend'] = merged_df_cleaned['est_weekend'].astype(int)
# merged_df_cleaned['est_jour_ferie'] = merged_df_cleaned['est_jour_ferie'].astype(int)

# 2. Remove quotes from jour_semaine if present and encode
merged_df_cleaned['jour_semaine'] = merged_df_cleaned['jour_semaine'].str.replace('"', '').str.strip()
le_jour = LabelEncoder()
merged_df_cleaned['jour_semaine_encoded'] = le_jour.fit_transform(merged_df_cleaned['jour_semaine'])

# 3. Label encode Gouvernerat
le_gouv = LabelEncoder()
merged_df_cleaned['Gouvernerat_encoded'] = le_gouv.fit_transform(merged_df_cleaned['Gouvernerat'])

# Save the mapping for interpretation
gouv_mapping = dict(zip(le_gouv.classes_, le_gouv.transform(le_gouv.classes_)))
print("Gouvernerat encoding mapping:", gouv_mapping)

# (Optional) See the mapping as a DataFrame
import pandas as pd
gouv_df = pd.DataFrame({'Gouvernerat': le_gouv.classes_, 'Gouvernerat_encoded': le_gouv.transform(le_gouv.classes_)})
print(gouv_df)

# Define the list of features to include in df_model
features_for_model = [
    'Caisse_Key',
    'Solde_Effet',
    'Montant_Bloque',
    'Solde_Mutiles',
    'Solde_Timbres',
    'Solde_TND',
    'Code_Agence',
    'date',
    'annee',
    'mois',
    'jour',
    'jour_semaine_encoded', # Use encoded column
    'lag_1',
    'lag_7',
    'rolling_mean_3',
    'rolling_std_7',
    'Gouvernerat_encoded' # Use encoded column
]


# Select only the features for df_model
df_model = merged_df_cleaned[features_for_model].copy()


print(df_model.head())

In [ ]:
df_model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort data by date
df_model = df_model.sort_values(by='date').reset_index(drop=True)

# Determine the split point for each Caisse_Key
train_test_split_indices = df_model.groupby('Caisse_Key')['date'].apply(
    lambda x: x.iloc[int(len(x) * 0.7)]
).reset_index(name='split_date')

# Merge the split dates back to the main dataframe
df_model = df_model.merge(train_test_split_indices, on='Caisse_Key', how='left')

# Split the data based on the split date for each Caisse_Key
df_train = df_model[df_model['date'] <= df_model['split_date']].copy()
df_test = df_model[df_model['date'] > df_model['split_date']].copy()

# Define features and target
features = [col for col in df_train.columns if col not in ['date', 'fact_Date_Position', 'Solde_TND', 'split_date']] # Exclude date columns, target, and split_date
target = 'Solde_TND'

X_train = df_train[features]
y_train = df_train[target]
X_test = df_test[features]
y_test = df_test[target]

# Drop the split_date column as it's no longer needed in the split dataframes
df_train = df_train.drop(columns=['split_date'])
df_test = df_test.drop(columns=['split_date'])
df_model = df_model.drop(columns=['split_date']) # Also drop from the original df_model


print("Training set shape:", df_train.shape)
print("Testing set shape:", df_test.shape)

# --- Plotting to verify the split ---

# Plot the number of rows per Caisse_Key in train and test sets
train_caisse_counts = df_train['Caisse_Key'].value_counts().sort_index()
test_caisse_counts = df_test['Caisse_Key'].value_counts().sort_index()

plt.figure(figsize=(15, 6))
plt.bar(train_caisse_counts.index, train_caisse_counts.values, label='Train')
plt.bar(test_caisse_counts.index, test_caisse_counts.values, label='Test', bottom=train_caisse_counts.values)
plt.xlabel('Caisse_Key')
plt.ylabel('Number of Data Points')
plt.title('Train/Test Split Distribution per Caisse_Key')
plt.legend()
plt.show()

# Plot the date range for a few sample Caisse_Keys
sample_caisses = df_model['Caisse_Key'].unique()[:5] # Take first 5 caisses

plt.figure(figsize=(15, 6))
for caisse_key in sample_caisses:
    caisse_data = df_model[df_model['Caisse_Key'] == caisse_key]
    plt.plot(caisse_data['date'], caisse_data['Solde_TND'], label=f'Caisse {caisse_key}')
    split_date = train_test_split_indices[train_test_split_indices['Caisse_Key'] == caisse_key]['split_date'].iloc[0]
    plt.axvline(split_date, color='red', linestyle='--', alpha=0.8) # Indicate split point with higher alpha

plt.xlabel('Date')
plt.ylabel('Solde_TND')
plt.title('Train/Test Split for Sample Caisse_Keys (Red line indicates split)')
plt.legend()
plt.show()

In [ ]:
print("Unique Caisse_Key in training set:", df_train['Caisse_Key'].nunique())
print("Unique Caisse_Key in testing set:", df_test['Caisse_Key'].nunique())

print("\nMaximum date in training set:", df_train['date'].max())
print("Maximum date in testing set:", df_test['date'].max())

print("\nMinimum date in training set:", df_train['date'].min())
print("Minimum date in testing set:", df_test['date'].min())

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ✅ Ensure annee, mois, jour are numeric (if they are numeric strings)
for col in ['annee', 'mois', 'jour']:
    if col in X_train.columns:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# Alternatively, if they should be treated as categorical:
# for col in ['annee', 'mois', 'jour']:
#     X_train[col] = X_train[col].astype('category')
#     X_test[col] = X_test[col].astype('category')

# ✅ Initialize and train the XGBoost model
model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# If you used .astype('category'), enable categorical handling:
# model = XGBRegressor(..., enable_categorical=True)

model.fit(X_train, y_train)

# ✅ Make predictions
y_pred_xgb = model.predict(X_test)

# ✅ Evaluate the model
print("XGBoost Model Evaluation:")
print("R² (Test):", r2_score(y_test, y_pred_xgb))
print("MAE (Test):", mean_absolute_error(y_test, y_pred_xgb))
print("RMSE (Test):", mean_squared_error(y_test, y_pred_xgb) ** 0.5)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Use the same features and target as before
X = df_train[features]
y = df_train[target]

# Use the same train/test split as before
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest R² (Test):", r2_score(y_test, y_pred_rf))
print("Random Forest MAE (Test):", mean_absolute_error(y_test, y_pred_rf))
print("Random Forest RMSE (Test):", mean_squared_error(y_test, y_pred_rf) ** 0.5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta

# Select a random Caisse_Key from df_model
random_caisse_key = np.random.choice(df_model['Caisse_Key'].unique())
# Filter the dataframe for the selected Caisse_Key
caisse_data_full = df_model[df_model['Caisse_Key'] == random_caisse_key].copy()

# Sort by date
caisse_data_full = caisse_data_full.sort_values(by='date')

# Get the last date and define the 2-month window
last_date_full = caisse_data_full['date'].max()
two_months_ago = last_date_full - pd.DateOffset(months=2)

# Filter for the last 2 months
caisse_data_last_2_months = caisse_data_full[caisse_data_full['date'] >= two_months_ago].copy()

# Prepare features
features_to_predict = caisse_data_last_2_months[features].copy()

# ✅ Convert all features to numeric to avoid object dtype issues
features_to_predict = features_to_predict.apply(pd.to_numeric, errors='coerce')

# Drop rows with NaNs
features_to_predict = features_to_predict.dropna()

# Make predictions if data is available
if not features_to_predict.empty:
    predicted_solde_xgb = model.predict(features_to_predict)

    # Align predictions
    caisse_data_last_2_months = caisse_data_last_2_months.loc[features_to_predict.index].copy()
    caisse_data_last_2_months['predicted_Solde_TND_xgb'] = predicted_solde_xgb

    # Plot only last 2 months
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=caisse_data_last_2_months, x='date', y='Solde_TND', label='Actual')
    sns.lineplot(data=caisse_data_last_2_months, x='date', y='predicted_Solde_TND_xgb', label='Predicted')

    plt.title(f'Actual vs Predicted Solde_TND for Caisse Key: {random_caisse_key} (Last 2 Months - xgb)')
    plt.xlabel('Date')
    plt.ylabel('Solde_TND')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"No complete feature rows available for Caisse Key {random_caisse_key} in the last 2 months.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta

# Select a random Caisse_Key from df_model
random_caisse_key = np.random.choice(df_model['Caisse_Key'].unique())
random_caisse_key = 476
# Filter the dataframe for the selected Caisse_Key
caisse_data_full = df_model[df_model['Caisse_Key'] == random_caisse_key].copy()

# Sort by date
caisse_data_full = caisse_data_full.sort_values(by='date')

# Get the last date and define the 2-month window
last_date_full = caisse_data_full['date'].max()
two_months_ago = last_date_full - pd.DateOffset(months=2)

# Filter for the last 2 months
caisse_data_last_2_months = caisse_data_full[caisse_data_full['date'] >= two_months_ago].copy()

# Prepare features
features_to_predict = caisse_data_last_2_months[features].copy()

# ✅ Convert all features to numeric (RF also needs numeric input)
features_to_predict = features_to_predict.apply(pd.to_numeric, errors='coerce')

# Drop rows with NaNs
features_to_predict = features_to_predict.dropna()

# Make predictions with RF if data is available
if not features_to_predict.empty:
    predicted_solde_rf = rf.predict(features_to_predict)

    # Align predictions
    caisse_data_last_2_months = caisse_data_last_2_months.loc[features_to_predict.index].copy()
    caisse_data_last_2_months['predicted_Solde_TND_RF'] = predicted_solde_rf

    # Plot only last 2 months
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=caisse_data_last_2_months, x='date', y='Solde_TND', label='Actual')
    sns.lineplot(data=caisse_data_last_2_months, x='date', y='predicted_Solde_TND_RF', label='Predicted (RF)')

    plt.title(f'Actual vs Predicted Solde_TND for Caisse Key: {random_caisse_key} (Last 2 Months - RF)')
    plt.xlabel('Date')
    plt.ylabel('Solde_TND')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"No complete feature rows available for Caisse Key {random_caisse_key} in the last 2 months.")


In [ ]:
import pandas as pd
import numpy as np

# Make a copy to not alter the original df_model
df_report = df_model.copy()

# --- Step 1: Determine train/test split per Caisse_Key ---
train_test_split_indices = df_report.groupby('Caisse_Key')['date'].apply(
    lambda x: x.iloc[int(len(x) * 0.7)]
).reset_index(name='split_date')

# Merge split dates into df_report
df_report = df_report.merge(train_test_split_indices, on='Caisse_Key', how='left')

# --- Step 2: Define features and target ---
features = [col for col in df_report.columns if col not in ['date', 'Solde_TND', 'split_date']]
target = 'Solde_TND'

# --- Step 3: Initialize columns ---
df_report["X_train"] = 0
df_report["X_test"] = 0
df_report["y_train"] = np.nan
df_report["y_test"] = np.nan
df_report["predicted_Solde_TND_RF"] = np.nan
df_report["predicted_Solde_TND_XGB"] = np.nan

# --- Step 4: Fill train/test flags and target ---
train_mask = df_report['date'] <= df_report['split_date']
test_mask = df_report['date'] > df_report['split_date']

df_report.loc[train_mask, "X_train"] = 1
df_report.loc[train_mask, "y_train"] = df_report.loc[train_mask, target]

df_report.loc[test_mask, "X_test"] = 1
df_report.loc[test_mask, "y_test"] = df_report.loc[test_mask, target]

# --- Step 5: Predict for all rows (RF + XGBoost) ---
df_features = df_report[features].apply(pd.to_numeric, errors='coerce')
valid_idx = df_features.dropna().index

# Random Forest prediction
df_report.loc[valid_idx, "predicted_Solde_TND_RF"] = rf.predict(df_features.loc[valid_idx])

# XGBoost prediction
df_report.loc[valid_idx, "predicted_Solde_TND_XGB"] = model.predict(df_features.loc[valid_idx])

# --- Step 6: Drop split_date column ---
df_report = df_report.drop(columns=['split_date'])

# --- Step 7: Save to Excel for Power BI ---
df_report.to_excel("df_model_report_powerbi_all_models.xlsx", index=False)
print("✅ df_report ready with RF and XGB predictions, train/test columns!")


In [ ]:
# --- Step 7: Save to CSV for Power BI ---
df_report.to_csv("df_model_report_powerbi_all_models.csv", index=False)
print("✅ df_report ready with RF and XGB predictions, train/test columns, saved as CSV!")
